# SMOTE + LLM Augmentation (Dokumentasi)

Kerangka kerja augmentasi data untuk mengatasi imbalance kelas pada dataset sentimen banjir (3 kelas: 0=negatif, 1=netral, 2=positif).

- **SMOTE** dipakai sebagai kerangka penetapan target: berapa sampel sintetis yang harus dibuat per kelas agar distribusi menjadi seimbang 1:1:1.
- **LLM (Gemini)** dipakai sebagai generator teks sintetis: setiap teks minoritas (netral/positif) dijadikan prompt untuk menghasilkan variasi kalimat baru yang natural, alih-alih interpolasi vektor yang merusak keterbacaan teks.
- Notebook ini murni **dokumentasi pipeline** dan memproduksi dataset augmented (`train_smote_llm.csv`) yang siap dipakai eksperimen berikutnya. Tidak ada training model di sini.

> Catatan konsistensi: tabel perbandingan di `temp_kernel_lora` (sel-67, `perbandingan_class_weight.csv`) memuat angka hardcoded placeholder (komentar `# sesuaikan hasil empiris`, `# ganti sesuai hasil aktual`) yang TIDAK cocok dengan hasil sel-28 dan sel-66 notebook yang sama. Angka tersebut tidak dipakai di sini.

In [ ]:
# =====================================================
# 0. INSTALL PACKAGE (internet on)
# =====================================================

!pip install -q google-genai

print("google-genai siap.")

In [ ]:
# =====================================================
# 0b. DIAGNOSA MOUNT INPUT
# =====================================================
import os

print("Isi /kaggle/input:")
if os.path.isdir("/kaggle/input"):
    for name in sorted(os.listdir("/kaggle/input")):
        path = os.path.join("/kaggle/input", name)
        print(f"  {name}/")
        if os.path.isdir(path):
            for f in sorted(os.listdir(path)):
                sub = os.path.join(path, f)
                print(f"    {f}/" if os.path.isdir(sub) else f"    {f}")
                if os.path.isdir(sub):
                    for g in sorted(os.listdir(sub)):
                        print(f"      {g}")
else:
    print("  /kaggle/input TIDAK ADA")

In [ ]:
# =====================================================
# 1. IMPORT LIBRARY
# =====================================================

import os
import json
import time
import random
import numpy as np
import pandas as pd

from google import genai  # google-genai

print("Library siap.")

In [ ]:
# =====================================================
# 2. SET SEED (replicability)
# =====================================================

seed = 42

random.seed(seed)
np.random.seed(seed)

print("Seed:", seed)

In [ ]:
# =====================================================
# 3. BACA DATA DARI KAGGLE DATASET
# =====================================================

import glob

# Path standar Kaggle: /kaggle/input/<dataset-name>/<file>
# Path alternatif (kernel non-GPU): /kaggle/input/datasets/<owner>/<dataset>/<file>
DATASET_DIR_CANDIDATES = [
    "/kaggle/input/thesis-indobert-processed-data",
    "/kaggle/input/datasets/emanuelembuaijdak/thesis-indobert-processed-data",
]

csv_path = None
for cand in DATASET_DIR_CANDIDATES:
    p = os.path.join(cand, "data_preprocessed_with_emoticon.csv")
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    # Fallback: cari file CSV secara rekursif di /kaggle/input
    hits = glob.glob("/kaggle/input/**/data_preprocessed_with_emoticon.csv", recursive=True)
    if hits:
        csv_path = hits[0]

if csv_path is None:
    raise FileNotFoundError(
        "data_preprocessed_with_emoticon.csv tidak ditemukan di /kaggle/input. "
        "Struktur /kaggle/input: " + str(os.listdir("/kaggle/input"))
    )

print("CSV ditemukan:", csv_path)

df = pd.read_csv(csv_path)

# Kolom wajib: text_bert (representasi BERT) dan label (0=negatif, 1=netral, 2=positif)
if "text_bert" not in df.columns:
    raise ValueError(f"Kolom 'text_bert' tidak ditemukan. Kolom: {df.columns.tolist()}")
if "label" not in df.columns:
    raise ValueError(f"Kolom 'label' tidak ditemukan. Kolom: {df.columns.tolist()}")

df["text_bert"] = df["text_bert"].fillna("").astype(str)

print("Kolom tersedia:", df.columns.tolist())
print("Total baris:", len(df))
print()
print("Distribusi label:")
print(df["label"].value_counts().sort_index())

In [ ]:
# =====================================================
# 4. KONFIGURASI API KEY
# =====================================================
# Prioritas: env var -> file secret di /kaggle/input (2 struktur mount).

import glob

SECRET_INPUT_DIRS = [
    "/kaggle/input",
    "/kaggle/input/datasets/emanuelembuaijdak",
]


def _read_secret_from_input(secret_names, pattern):
    for base in SECRET_INPUT_DIRS:
        for name in secret_names:
            for path in glob.glob(f"{base}/{name}/*"):
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        content = f.read().strip()
                except Exception:
                    continue
                if pattern(content):
                    return content
    return None


def _read_secret_json():
    for base in SECRET_INPUT_DIRS:
        for name in ["gemini-secret", "secrets", "geminikey", "kaggle-secrets"]:
            try:
                with open(f"{base}/{name}/secret.json", "r", encoding="utf-8") as f:
                    data = json.load(f)
                api_key = data.get("GEMINI_API_KEY") or data.get("GOOGLE_API_KEY")
                if api_key:
                    return api_key
            except Exception:
                continue
    return None


api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

if not api_key:
    # Format file key: satu baris berisi kunci, atau JSON {"GEMINI_API_KEY": "..."}
    api_key = _read_secret_from_input(
        ["gemini-secret", "secrets", "geminikey", "kaggle-secrets"],
        lambda c: len(c) > 10 and " " not in c,
    )

if not api_key:
    api_key = _read_secret_json()

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY tidak ditemukan. Set env var, atau unggah dataset secret"
        " (nama folder: gemini-secret, berisi file secret.json / file kunci)."
    )

print("API key terbaca (panjang):", len(api_key))
print("Prefix:", api_key[:8] + "...")

In [ ]:
# =====================================================
# 5. TARGET JUMLAH PER KELAS (KERANGKA SMOTE)
# =====================================================
# Rasio target 1:1:1 -> semua kelas diset ke jumlah kelas mayoritas.

label_counts = df["label"].value_counts().sort_index()

target_per_class = int(label_counts.max())

targets = {
    0: target_per_class,
    1: target_per_class,
    2: target_per_class,
}

print("Jumlah aktual per kelas:")
for label in [0, 1, 2]:
    print(f"  Kelas {label}: {int(label_counts[label])}")

print("\nTarget 1:1:1 per kelas:")
for label in [0, 1, 2]:
    print(f"  Kelas {label}: {targets[label]}")

print("\nJumlah sintetis yang harus di-generate:")
for label in [0, 1, 2]:
    need = targets[label] - int(label_counts[label])
    print(f"  Kelas {label}: {need}")

In [ ]:
# =====================================================
# 6. DEFINISI PROMPT (BAHASA INDONESIA, NETRAL)
# =====================================================

PROMPT_TEMPLATE = """\
Kamu adalah asisten peneliti sentimen bencana banjir di Indonesia.
Buatlah {n} kalimat TWEET BARU dalam bahasa Indonesia tentang banjir yang
memiliki sentimen {sentimen} (sama persis dengan contoh di bawah).

ATURAN WAJIB:
1. Setiap kalimat harus BARU dan BERBEDA dari contoh maupun antar kalimat.
2. Jangan mengulang contoh kata demi kata; ubah struktur, sudut pandang, atau konteks
   (mis. wilayah, waktu, dampak, tanggapan warga/instansi).
3. Kalimat harus natural seperti tweet asli (boleh singkatan, tidak harus baku),
   TETAPI harus tetap jelas topiknya: banjir, dampaknya, atau tanggapannya.
4. JANGAN menambahkan angka, label, atau simbol penanda di awal/akhir kalimat.
5. Keluarkan tepat {n} kalimat, masing-masing dalam satu baris, tanpa nomor,
   tanpa tanda kutip, tanpa bullet.

Contoh kalimat dengan sentimen {sentimen}:
{examples}
"""

SENTIMEN_LABEL = {0: "negatif", 1: "netral", 2: "positif"}

print("Prompt template siap.")

In [ ]:
# =====================================================
# 7. FUNGSI GENERASI LLM (GEMINI)
# =====================================================

from google.genai import types

client = genai.Client(api_key=api_key)

GEMINI_MODEL = "gemini-2.0-flash"


def parse_generated_lines(text):
    """Ambil baris non-kosong dari output model."""
    lines = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        # Buang penanda umum yang mungkin muncul
        line = line.lstrip("-*#0123456789. )\"'")
        line = line.rstrip('\"')
        if line:
            lines.append(line)
    return lines


def generate_batch(examples, sentimen, n, max_retries=3):
    """Generate n kalimat baru untuk satu kelas."""
    prompt = PROMPT_TEMPLATE.format(
        n=n,
        sentimen=sentimen,
        examples="\n".join(f"- {t}" for t in examples),
    )

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.9,
                    max_output_tokens=8192,
                ),
            )
            text = response.text or ""
            lines = parse_generated_lines(text)
            if lines:
                return lines
        except Exception as exc:
            print(f"  [retry {attempt}/{max_retries}] {type(exc).__name__}: {exc}")
            time.sleep(5 * attempt)

    return []


print("Fungsi generasi siap. Model:", GEMINI_MODEL)

In [ ]:
# =====================================================
# 8. PIPELINE GENERASI PER KELAS MINORITAS
# =====================================================
# Untuk setiap kelas minoritas: ambil contoh asli sebagai referensi prompt,
# generate dalam batch kecil (BATCH_GEN=10) hingga target terpenuhi.

BATCH_GEN = 10
MAX_ITER = 50  # pengaman agar loop tidak tak terbatas

augmented_rows = []

for label in [1, 2]:  # netral, positif
    need = targets[label] - int(label_counts[label])

    if need <= 0:
        print(f"Kelas {label} sudah cukup, lewati.")
        continue

    print(f"\n=== Kelas {label} ({SENTIMEN_LABEL[label]}) - butuh {need} ===")

    pool = df[df["label"] == label]["text_bert"].tolist()
    rng = random.Random(seed + label)
    rng.shuffle(pool)

    collected = []
    seen = set()
    iter_idx = 0

    while len(collected) < need and iter_idx < MAX_ITER:
        iter_idx += 1

        # Contoh referensi: 3 teks acak dari kelas ini
        examples = rng.sample(pool, k=min(3, len(pool)))

        batch_size = min(BATCH_GEN, need - len(collected))
        lines = generate_batch(
            examples=examples,
            sentimen=SENTIMEN_LABEL[label],
            n=batch_size,
        )

        added = 0
        for line in lines:
            key = line.lower()
            if key in seen or key in set(pool):
                continue  # duplikat
            if len(line) < 10:
                continue  # terlalu pendek
            seen.add(key)
            collected.append(line)
            added += 1
            if len(collected) >= need:
                break

        print(
            f"  iter {iter_idx}: request {batch_size}, dapat {added}, "
            f"total {len(collected)}/{need}"
        )

        time.sleep(1)  # jaga rate limit

    if len(collected) < need:
        print(f"  [Peringatan] Kelas {label} hanya terkumpul {len(collected)}/{need}")

    for text in collected:
        augmented_rows.append({
            "text_bert": text,
            "label": label,
            "source": "llm_synthetic",
        })

    print(f"Kelas {label} selesai: {len(collected)} teks sintetis.")

print("\nPipeline generasi selesai.")

In [ ]:
# =====================================================
# 9. GABUNGKAN DATA ORISINIL + SINTETIS
# =====================================================

df_aug = pd.concat([
    df[["text_bert", "label"]].assign(source="original"),
    pd.DataFrame(augmented_rows),
], ignore_index=True)

print("Distribusi setelah augmentasi:")
print(df_aug["label"].value_counts().sort_index())

print("\nJumlah per source:")
print(df_aug["source"].value_counts())

In [ ]:
# =====================================================
# 10. SIMPAN HASIL
# =====================================================

OUTPUT_CSV = "train_smote_llm.csv"

df_aug.to_csv(OUTPUT_CSV, index=False)

print(f"Tersimpan: /kaggle/working/{OUTPUT_CSV}")
print("Total baris:", len(df_aug))